In [ ]:
import pandas as pd
import os
from pathlib import Path
import csv
import openpyxl

os.chdir('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity')

In [36]:
# Define dataset specifications
dataset_specs = {
    "obis_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\OBIS\obis_final.parquet"),
    },
    "gbif_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\GBIF\gbif_final.parquet"),
    },
    "ogsl_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\OGSL\ogsl_final.parquet"),
    },
    "cb_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\CB\cb_final.parquet"),
    },
    "au_final": {
        "parquet": Path(r"C:\Users\hamod4691\Code\arctic-benthic-biodiversity\AU\au_final.parquet"),
    },
}
# Load all parquet sources
parquets = {}
for dataset, spec in dataset_specs.items():
    parquet_path = spec["parquet"]
    if parquet_path.exists():
        df = pd.read_parquet(parquet_path)
        parquets[dataset] = df
# Merge all datasets
all_columns = set()
for df in parquets.values():
    all_columns.update(df.columns)
col_order = sorted(all_columns)
merged = pd.concat([parquets[d] for d in parquets.keys()], ignore_index=True, sort=False)[col_order]
# Load column data type specifications from Excel
dtype_specs = pd.read_excel("column_data_types.xlsx")
dtype_mapping = dict(zip(dtype_specs["column"], dtype_specs["final"]))
# Harmonize data types based on the specifications
for col, target_dtype in dtype_mapping.items():
    if col not in merged.columns:
        merged[col] = pd.Series([pd.NA] * len(merged), index=merged.index)
    if target_dtype == "string":
        merged[col] = merged[col].astype("string")
    elif target_dtype == "Int64":
        merged[col] = pd.to_numeric(merged[col], errors="coerce").astype("Int64")
    elif target_dtype == "boolean":
        merged[col] = merged[col].astype("boolean")
    elif target_dtype == "float64":
        merged[col] = pd.to_numeric(merged[col], errors="coerce").astype("float64")
# Save the harmonized merged dataset
merged_path = Path("All_data/merged_all.parquet")
merged.to_parquet(merged_path, index=False)
print(f"Saved merged dataset to: {merged_path}")

Saved merged dataset to: All_data\merged_all.parquet


In [37]:
all_merged = pd.read_parquet("All_data/merged_all.parquet")

# Column type overview
column_types = pd.DataFrame({
    "column": all_merged.columns,
    "dtype": all_merged.dtypes.astype(str).values,
    "non_null_count": all_merged.notna().sum().values,
    "null_count": all_merged.isna().sum().values,
    "non_null_pct": (all_merged.notna().sum().values / len(all_merged)) * 100,
})

column_types = column_types.sort_values(
    by=["non_null_count", "column"], ascending=[False, True]
 )

column_types.to_csv("All_data/merged_all_column_types.csv", index=False)

column_types.head(20)

,column,dtype,non_null_count,null_count,non_null_pct
11,datasetKey,string,4556292,0,100.000000
59,verbatimScientificName,string,4555624,668,99.985339
13,decimalLatitudeStart,float64,4555380,912,99.979984
15,decimalLongitudeStart,float64,4555380,912,99.979984
49,sourceID,string,4549035,7257,99.840726
1,acceptedScientificName,string,4531324,24968,99.452011
48,sourceDatabase,string,4531324,24968,99.452011
44,phylum,string,4531255,25037,99.450496
57,verbatimDate,string,4428923,127369,97.204547
9,class,string,4413411,142881,96.864095


# Filter out irrelevant datasets

Go through datasets with missing classification manually, save as datasets_GBIF_OBIS_manual.xlsx

Key in Keep column

In [38]:
# Load the datasets
manual_path = Path("All_data/datasets_GBIF_OBIS_manual.xlsx")
merged_path = Path("All_data/merged_all.parquet")

manual = pd.read_excel(manual_path)
merged = pd.read_parquet(merged_path)

# Normalize datasetKey in both for matching
def _norm_series(series):
    """Normalize a series by stripping whitespace, converting to lowercase, and replacing empty strings with NA"""
    return (
        series.astype('string')
              .str.strip()
              .str.lower()
              .replace({'': pd.NA})
    )

manual['datasetKey_norm'] = _norm_series(manual['alternateIdentifier'])
merged['datasetKey_norm'] = _norm_series(merged['datasetKey'])

# Merge the Keep column from manual
merged = merged.merge(
    manual[['datasetKey_norm', 'Keep']],
    on='datasetKey_norm',
    how='left'
)

# Drop the temporary normalized column
merged = merged.drop(columns=['datasetKey_norm'])

# Save back to the same location
merged.to_parquet(merged_path, index=False)
print(f"Saved updated file to: {merged_path}")

Saved updated file to: All_data\merged_all.parquet


Remove records with R or U

In [39]:
# Load the deduplicated dataset
merged = pd.read_parquet(merged_path)

# Filter out rows where Keep column contains 'R' or 'U'
merged_filtered = merged[~merged['Keep'].isin(['R', 'U'])].copy()

print(f"Original rows: {len(merged)}")
print(f"Removed rows (Keep='R' or 'U'): {len(merged) - len(merged_filtered)}")
print(f"Remaining rows: {len(merged_filtered)}")

# Save back to the same location
merged_filtered.to_parquet(merged_path, index=False)
print(f"Saved filtered file to: {merged_path}")

Original rows: 4556292
Removed rows (Keep='R' or 'U'): 370103
Remaining rows: 4186189
Saved filtered file to: All_data\merged_all.parquet


# Reapply taxonomic filter 

This is just a first step as it only targets the actual level of identification

In [40]:
merged_path = Path("All_data/merged_all.parquet")
merged = pd.read_parquet(merged_path)

# Define aphiaIDs to remove
aphia_ids_to_remove = [
    196109, 1490567, 2689, 10305, 10313, 10329, 11014, 10331, 10211, 325345, 103353, 447, 10304, 17931, 10309, 10326, 16349,
    10295, 177531, 10306, 10297, 154206, 154156, 10308, 148390, 10316, 1600607, 10294, 160002, 11730, 10322, 1600798, 10327,
    10321, 10919, 10288, 10310, 10320, 767376, 10296, 10314, 10332, 991168, 154219, 397228, 151811, 120239, 154217, 151956, 1600606,
    10323, 10315, 833292, 154157, 10317, 152241, 833022, 123096, 833089, 154244, 10319, 154571, 1600603, 10318, 484684, 715608,
    10201, 1374984, 1100, 1101, 345938, 22597, 1106, 22598, 22605, 1035042, 1104, 1128, 153726, 845456, 160326,
    1370, 16350, 135225,
    101417, 101420, 101418, 101422, 236594, 101415, 196118, 158321, 110671, 154884, 104077, 104079, 104084, 104087, 104092, 128586,
    345271, 5953, 101424, 104103, 104104, 958, 227115, 244471, 234095, 430454, 732565, 834811, 158103, 151708, 160343, 826205, 153908,
    248310, 443294, 489101, 248325,
    129715, 129329, 206129, 129722, 104108, 104111, 117212, 104137, 458856, 125815, 1434994, 117107, 104115,
    104193, 104159, 104116, 104117, 127533, 117216, 101602, 148370, 135361, 122429, 104174, 105399, 110673, 103358, 104121, 117111,
    104183, 104190, 104164, 115341, 103367, 104201, 196874, 104165, 104126, 105410, 104228, 104229, 104236, 104149, 110679, 128691,
    104204, 227368, 1296320, 181554, 575798, 955416, 149909, 342551, 1040034, 137356, 342556, 342562, 1062020, 153384,
    468526, 1590744, 150155, 160018, 422842, 342573, 833124, 1590943, 422845, 137388, 347529, 769713, 227271, 342605, 422869, 1372207,
    176320, 157596, 475530, 1255793, 137359, 342614, 137360, 994225, 157599, 827473, 395102,
    104251, 104257, 104267, 254600, 104270, 104272, 117849, 157611, 104372, 104375, 265150, 127766, 127767, 148379, 117615,
    117616, 117333, 146735, 104283, 104464, 104465, 104467, 104496, 104284, 104287, 346271, 350046, 346272, 104290, 104292, 346278,
    139178, 146852, 150191, 183084, 183086, 285347, 117855, 135414, 135301, 102553, 135418, 346375, 127782, 149756, 104396, 196775,
    220851, 105414, 105416, 254621, 103375, 266042, 346483, 254602, 237965, 104316, 104321, 196593, 104422, 346520, 104578, 104579,
    117623, 135900, 107590, 107591, 103251, 103253, 247787, 325370, 353202, 254606, 104593, 353211, 220910, 346637, 117742, 110690,
    106377, 104632, 104633, 196784, 157675, 104513, 116115, 346683, 348210, 104841, 353500, 715130, 135442, 104470, 1770057, 103417,
    103413, 254722, 106642, 106656, 129110, 149765, 344717, 128930, 128931, 128935, 149767, 128944, 128946, 344725, 128948, 104694,
    104557, 104560, 104561, 104566, 104568, 117457, 254603, 130219, 135294, 362050, 338262, 117458, 1565062, 104514, 104515, 104516,
    104517, 104339, 105445, 254609, 117989, 105448, 105457, 254610, 104790, 104798, 104799, 254611, 103278, 104811, 720272, 117866,
    117267, 104846, 254612, 104848, 254613, 104851, 104852, 104853, 104855, 104856, 104878, 104459, 103259, 156451, 325384, 110708,
    110709, 110711, 131554, 128956, 128957, 128960, 131582, 104701, 360768, 423829, 104713, 423830, 102166, 102297, 102644, 102647,
    1765892, 139662, 1412088, 254615, 1040959, 137398, 1043400, 1665457, 128183, 1073463, 1320683, 460821, 157637, 460942, 423307,
    461022, 423320, 461032, 1402051, 1054492, 234074, 160017, 1372692, 1073916, 1071120, 1070114, 423472, 140184, 423498, 706613,
    1346539, 1068860, 1070488, 1068806, 1348014, 1070490, 1068807, 1070491, 1070492, 730406, 444591, 137480, 1510149, 1412563,
    423638, 1406479, 1642842, 156389, 423752, 1040983, 264152
]

# Remove duplicates from the list
aphia_ids_to_remove = list(set(aphia_ids_to_remove))

# Count before
count_before = len(merged)
print(f"Records before: {count_before}")

# Filter out records with aphiaID in the removal list
merged = merged[~merged['aphiaID'].isin(aphia_ids_to_remove)].copy()

# Count after
count_after = len(merged)
records_removed = count_before - count_after

print(f"Records after: {count_after}")
print(f"Records removed: {records_removed}")
merged.to_parquet(merged_path, index=False)

Records before: 4186189
Records after: 4185553
Records removed: 636


# Missing aphiaIDs

In [7]:
# List unique acceptedScientificName entries (with phylum/class/order/family/genus where available),
# and include acceptedTaxonKey and sourceDatabase when available

merged_path = Path("All_data/merged_all.parquet")
merged = pd.read_parquet(merged_path)

# helper to find column by substring
def find_col(substr):
    substr = substr.lower()
    for c in merged.columns:
        if substr in c.lower():
            return c
    return None

acc_col = find_col("acceptedScientificName") or find_col("acceptedname")
aphia_col = find_col("aphiaID")
tax_ranks = ["phylum", "class", "order", "family", "genus"]
tax_cols = {r: find_col(r) for r in tax_ranks}
# locate optional fields
taxonkey_col = find_col("taxonkey")
source_col = find_col("sourcedatabase") or find_col("source")

if acc_col is None:
    raise ValueError("Could not find a scientific name column in the dataset.")
if aphia_col is None:
    raise ValueError("Could not find an aphiaID column in the dataset.")

# mask for missing aphiaID (NaN or empty)
missing_aphia_mask = merged[aphia_col].isna() | (merged[aphia_col].astype("string").str.strip() == "")

subset = merged.loc[missing_aphia_mask, :]

# helper to normalise and collapse unique non-null values
def collapse_unique(series):
    s = series.astype("string").str.strip().replace({"": pd.NA}).dropna().unique().tolist()
    return "; ".join(sorted(s)) if s else pd.NA

agg_dict = {acc_col: "size"}
for r, c in tax_cols.items():
    if c is not None:
        agg_dict[c] = collapse_unique

# perform grouping
grouped = (
    subset.groupby(acc_col)
          .agg(count=("index" if "_dummy" in merged.columns else acc_col, "size"))
)

# build DataFrame with taxonomies and optional fields
rows = []
for name, grp in subset.groupby(acc_col):
    row = {"acceptedScientificName": name, "count": len(grp)}
    for rank, col in tax_cols.items():
        row[rank] = collapse_unique(grp[col]) if col is not None else pd.NA
    # optional acceptedTaxonKey and sourceDatabase
    row['acceptedTaxonKey'] = collapse_unique(grp[taxonkey_col]) if taxonkey_col is not None else pd.NA
    row['sourceDatabase'] = collapse_unique(grp[source_col]) if source_col is not None else pd.NA
    rows.append(row)

result = pd.DataFrame(rows).sort_values("count", ascending=False)

out_path = Path("All_data/gbif_missing_aphiaID.xlsx")
result.to_excel(out_path, index=False)
print(f"Saved {len(result)} names to: {out_path}")

Saved 15079 names to: All_data\gbif_missing_aphiaID.xlsx


Go through and fill in manually (extract from current coverage in gbif_missing_aphiaID_manual.xlsx)

# Fill in the missing GBIF aphiaIDs

In [9]:
# Load the merged dataset and the manual aphiaID file
merged = pd.read_parquet(merged_path)
manual_aphia = pd.read_excel("All_data/gbif_missing_aphiaID_manual.xlsx")

print(f"Merged dataset rows: {len(merged)}")
print(f"Manual aphiaID entries: {len(manual_aphia)}")

# Use acceptedTaxonKey as the key to merge
# First, ensure acceptedTaxonKey is comparable (convert to string and strip)
merged['acceptedTaxonKey'] = merged['acceptedTaxonKey'].astype('string').str.strip()
manual_aphia['acceptedTaxonKey'] = manual_aphia['acceptedTaxonKey'].astype('string').str.strip()

# Ensure aphiaID columns are a consistent nullable integer type to avoid mixed object types
if 'aphiaID' in merged.columns:
    merged['aphiaID'] = pd.to_numeric(merged['aphiaID'], errors='coerce').astype('Int64')
else:
    merged['aphiaID'] = pd.Series([pd.NA] * len(merged), index=merged.index).astype('Int64')

if 'aphiaID' in manual_aphia.columns:
    manual_aphia['aphiaID'] = pd.to_numeric(manual_aphia['aphiaID'], errors='coerce').astype('Int64')
else:
    manual_aphia['aphiaID'] = pd.Series([pd.NA] * len(manual_aphia), index=manual_aphia.index).astype('Int64')

# Count aphiaID blanks before
blanks_before = merged['aphiaID'].isna().sum()

# Merge on acceptedTaxonKey, keeping all rows from merged
merged = merged.merge(
    manual_aphia[['acceptedTaxonKey', 'aphiaID', 'flagged', 'flaggedComment']],
    on='acceptedTaxonKey',
    how='left',
    suffixes=('_old', '')
)

# Fill in missing values in the original columns with the new ones
# Coerce any merged-back old columns to Int64 before filling to keep dtype consistent
if 'aphiaID_old' in merged.columns:
    merged['aphiaID_old'] = pd.to_numeric(merged['aphiaID_old'], errors='coerce').astype('Int64')
    merged['aphiaID'] = merged['aphiaID'].fillna(merged['aphiaID_old']).astype('Int64')
    merged = merged.drop(columns=['aphiaID_old'])

if 'flagged_old' in merged.columns:
    merged['flagged'] = merged['flagged'].fillna(merged['flagged_old'])
    merged = merged.drop(columns=['flagged_old'])

if 'flaggedComment_old' in merged.columns:
    merged['flaggedComment'] = merged['flaggedComment'].fillna(merged['flaggedComment_old'])
    merged = merged.drop(columns=['flaggedComment_old'])

# Count aphiaID blanks after
blanks_after = merged['aphiaID'].isna().sum()
aphiaid_added = blanks_before - blanks_after

print(f"\nAphiaID blanks before: {blanks_before}")
print(f"AphiaID blanks after: {blanks_after}")
print(f"AphiaID entries added: {aphiaid_added}")

# Save back to the same location (ensure aphiaID remains Int64)
merged['aphiaID'] = merged['aphiaID'].astype('Int64')
merged.to_parquet(merged_path, index=False)
print(f"\nSaved updated file to: {merged_path}")

Merged dataset rows: 4186189
Manual aphiaID entries: 380

AphiaID blanks before: 2563334
AphiaID blanks after: 2556546
AphiaID entries added: 6788


Merged dataset rows: 4186189
Manual aphiaID entries: 380

AphiaID blanks before: 2563334
AphiaID blanks after: 2556546
AphiaID entries added: 6788


ArrowTypeError: ("Expected bytes, got a 'bool' object", 'Conversion failed for column flagged with type object')

# Remove BOLD names

In [ ]:
merged = pd.read_parquet(merged_path)

before_count = len(merged)

mask_bold = (
    merged["acceptedScientificName"]
    .astype("string")
    .str.strip()
    .str.startswith("BOLD:", na=False)
)

removed_count = int(mask_bold.sum())
merged = merged.loc[~mask_bold].copy()

merged.to_parquet(merged_path, index=False)

print(f"Removed records: {removed_count}")
print(f"Remaining records: {len(merged)}")
print(f"Saved updated file to: {merged_path}")

Removed records: 15580
Remaining records: 3988030
Saved updated file to: All_data\merged_all.parquet


# Remove acceptedTaxonKey labeled "x"

In [ ]:
# Load the merged dataset and the manual aphiaID file
merged = pd.read_parquet(merged_path)
manual_aphia = pd.read_excel("All_data/gbif_missing_aphiaID_manual.xlsx")

# Filter to only rows where remove column is 'x'
remove_keys = manual_aphia[manual_aphia['remove'] == 'x']['acceptedTaxonKey'].astype('string').str.strip().unique()

print(f"Merged dataset rows before: {len(merged)}")
print(f"acceptedTaxonKeys marked for removal: {len(remove_keys)}")

# Remove records with acceptedTaxonKey in the remove list
merged = merged[~merged['acceptedTaxonKey'].astype('string').str.strip().isin(remove_keys)].copy()

records_removed = len(merged_path.read_parquet()) - len(merged) if hasattr(merged_path, 'read_parquet') else 0
# More reliable count
before_count = pd.read_parquet(merged_path).shape[0]
after_count = len(merged)
records_removed = before_count - after_count

print(f"Merged dataset rows after: {after_count}")
print(f"Records removed: {records_removed}")

# Save back to the same location
merged.to_parquet(merged_path, index=False)
print(f"Saved updated file to: {merged_path}")

Merged dataset rows before: 3988030
acceptedTaxonKeys marked for removal: 133
Merged dataset rows after: 3985257
Records removed: 2773
Saved updated file to: All_data\merged_all.parquet


# aphiaID coverage

In [ ]:
# Coverage (%) of aphiaID per sourceDatabase
src = source_col if 'source_col' in globals() else 'sourceDatabase'
aph = aphia_col if 'aphia_col' in globals() else 'aphiaID'

s = merged[src].astype('string').str.strip().replace({'': pd.NA})
a = merged[aph].astype('string').str.strip().replace({'': pd.NA})

coverage = (
    pd.DataFrame({'sourceDatabase': s, 'aphiaID': a})
      .dropna(subset=['sourceDatabase'])                      # only named sources
      .groupby('sourceDatabase', observed=True)
      .agg(total=('aphiaID', 'size'),
           non_null=('aphiaID', lambda x: x.notna().sum()))
      .reset_index()
)
coverage['coverage_percent'] = (coverage['non_null'] / coverage['total']) * 100
coverage = coverage.sort_values('coverage_percent', ascending=False)

coverage.to_csv("aphiaID_coverage_by_source.csv", index=False)
print(coverage)

  sourceDatabase    total  non_null  coverage_percent
0    CRITTERBASE   243126    243126             100.0
1           GBIF  2386556   2386556             100.0
2           OBIS  1348325   1348325             100.0
3           OGSL     7250      7250             100.0


# Unique aphiaIDs

In [81]:
output_dir = Path("All_data")
output_dir.mkdir(parents=True, exist_ok=True)

merged_path = Path("All_data/merged_all.parquet")
merged = pd.read_parquet(merged_path)

unique_aphiaids = (
    pd.to_numeric(merged["aphiaID"], errors="coerce")
      .dropna()
      .astype("int64")
      .drop_duplicates()
      .sort_values()
      .to_frame(name="aphiaID")
)

output_path = output_dir / "unique_aphiaIDs.csv"
unique_aphiaids.to_csv(output_path, index=False)

print(f"Saved {len(unique_aphiaids)} unique aphiaID values to: {output_path}")

Saved 13141 unique aphiaID values to: All_data\unique_aphiaIDs.csv


# Move the section below

# Deduplicate

Between-database deduplication

In [ ]:
# Create a source priority mapping
source_priority = {
    'OGSL': 1,
    'CRITTERBASE': 2,
    'OBIS': 3,
    'GBIF': 4
}

# Deduplication key columns (we'll use rounded coords ≈50 m precision)
key_cols = ['year', 'startDayOfYear', 'decimalLongitudeStart_round', 'decimalLatitudeStart_round', 'aphiaID']

# --- Preprocess / normalize fields so "missing" is consistent ---
# Work on a copy and attach a stable row id for reliable keep/remove tracing
merged = merged.copy()
if '_row_id' not in merged.columns:
    merged['_row_id'] = merged.index.astype('int64')

# Normalize sourceDatabase (strip/case)
merged['sourceDatabase_norm'] = (
    merged['sourceDatabase'].astype('string')
          .str.strip()
          .str.upper()
)
merged['sourceDatabase_norm'] = merged['sourceDatabase_norm'].replace({'': pd.NA, 'NAN': pd.NA})

# Normalize numeric key columns (coerce invalid values to NaN)
for num_col in ['year', 'startDayOfYear', 'decimalLongitudeStart', 'decimalLatitudeStart', 'aphiaID']:
    if num_col in merged.columns:
        merged[num_col] = pd.to_numeric(merged[num_col], errors='coerce')

# Compute rounded coordinates at ~50 m precision
import numpy as np
# Approximate meters per degree latitude
_m_per_deg_lat = 111320.0
_lat_res_deg = 50.0 / _m_per_deg_lat  # ~0.000449 degrees (~50 m in latitude)

# For longitude, resolution depends on latitude (meters per degree longitude = 111320 * cos(lat))
# Avoid division by zero near poles by clipping cos(lat)
lat_rad = np.deg2rad(merged['decimalLatitudeStart'].astype('float').fillna(0.0))
_lon_res_deg = 50.0 / ( _m_per_deg_lat * np.clip(np.abs(np.cos(lat_rad)), 1e-6, 1.0) )

# Rounded latitude (same resolution for all rows)
merged['decimalLatitudeStart_round'] = (merged['decimalLatitudeStart'] / _lat_res_deg).round() * _lat_res_deg

# Rounded longitude (row-wise resolution)
merged['decimalLongitudeStart_round'] = (merged['decimalLongitudeStart'] / _lon_res_deg).round() * _lon_res_deg

# Identify rows with all key deduplication columns non-null (these are eligible for dedup)
required_numeric_cols = ['year', 'startDayOfYear', 'aphiaID']
required_text_cols = ['sourceDatabase_norm']
# plus the rounded coords must be present
numeric_complete = merged[required_numeric_cols].notna().all(axis=1)
coords_complete = merged[['decimalLongitudeStart_round', 'decimalLatitudeStart_round']].notna().all(axis=1)
text_complete = merged[required_text_cols].apply(
    lambda s: s.astype('string').str.strip().notna() & (s.astype('string').str.strip() != '')
).all(axis=1)
within_mask = numeric_complete & coords_complete & text_complete

# Split into rows to deduplicate and rows to keep as-is
to_dedup = merged[within_mask].copy()
to_keep = merged[~within_mask].copy()

# For each place+date combination, count how many distinct databases have it
place_date_db_count = to_dedup.groupby(['year', 'aphiaID', 'startDayOfYear', 'decimalLongitudeStart_round', 'decimalLatitudeStart_round'])['sourceDatabase_norm'].nunique().reset_index()
place_date_db_count.columns = ['year', 'aphiaID', 'startDayOfYear', 'decimalLongitudeStart_round', 'decimalLatitudeStart_round', 'db_count']

# Merge this count back into to_dedup
to_dedup = to_dedup.merge(place_date_db_count, on=['year', 'aphiaID', 'startDayOfYear', 'decimalLongitudeStart_round', 'decimalLatitudeStart_round'], how='left')

# Only rows where place+date appears in MULTIPLE databases are eligible for removal
cross_db_duplicates = to_dedup[to_dedup['db_count'] > 1].copy()
within_db_only = to_dedup[to_dedup['db_count'] == 1].copy()

if len(cross_db_duplicates) > 0:
    # For cross-database duplicates, keep all rows from the highest-priority database per place+date
    cross_db_duplicates['_priority'] = (
        cross_db_duplicates['sourceDatabase_norm'].map(source_priority).fillna(999)
    )
    # Keep sorted view for debugging readability
    cross_db_duplicates_sorted = cross_db_duplicates.sort_values(by=['year', 'aphiaID', 'startDayOfYear', 'decimalLongitudeStart_round', 'decimalLatitudeStart_round', '_priority'], na_position='last')

    # Compute best priority within each place+date group and keep all rows at that priority
    best_priority = cross_db_duplicates.groupby(['year', 'aphiaID', 'startDayOfYear', 'decimalLongitudeStart_round', 'decimalLatitudeStart_round'])['_priority'].transform('min')
    cross_db_duplicates_deduped = cross_db_duplicates[cross_db_duplicates['_priority'] == best_priority].copy()
else:
    cross_db_duplicates_deduped = cross_db_duplicates.copy()

# Within-database duplicates (same place/date in only one database) are kept as-is
within_db_only = within_db_only.copy()

# Compute which rows were removed during cross-database deduplication using stable row ids
if len(cross_db_duplicates) > 0:
    kept_row_ids = set(cross_db_duplicates_deduped['_row_id'].tolist())
    removed = cross_db_duplicates[~cross_db_duplicates['_row_id'].isin(kept_row_ids)].copy()
else:
    removed = cross_db_duplicates.copy()

# Drop helper columns before final concat/save
drop_cols = [c for c in ['_priority', 'db_count'] if c in cross_db_duplicates_deduped.columns]
cross_db_duplicates_deduped = cross_db_duplicates_deduped.drop(columns=drop_cols)
within_db_only = within_db_only.drop(columns=['db_count'] if 'db_count' in within_db_only.columns else [])

# Combine deduplicated cross-database records with within-database records
to_dedup_deduped = pd.concat([cross_db_duplicates_deduped, within_db_only], ignore_index=True)

# Concatenate back together
merged_deduped = pd.concat([to_dedup_deduped, to_keep], ignore_index=True)

# Do not persist helper row id
if '_row_id' in merged_deduped.columns:
    merged_deduped = merged_deduped.drop(columns=['_row_id'])

print(f"Original merged dataset: {len(merged)} rows")
print(f"After deduplication: {len(merged_deduped)} rows")
print(f"Rows deduplicated: {len(merged) - len(merged_deduped)}")
print(f"Cross-database duplicates found: {len(cross_db_duplicates)}")
print(f"Within-database records preserved: {len(within_db_only)}")

# Save the deduplicated dataset
merged_deduped.to_parquet('All_data/merged_all_deduplicated.parquet', index=False)
print(f"Saved deduplicated dataset to: All_data/merged_all_deduplicated.parquet")

if len(cross_db_duplicates) > 0:
    removed_counts = removed['sourceDatabase_norm'].value_counts(dropna=False).rename_axis('sourceDatabase').reset_index(name='removed_count')
    print('\nRemoved counts by sourceDatabase:')
    print(removed_counts)
    removed_counts.to_csv('dedup_removed_by_sourceDatabase.csv', index=False)
    print('Saved removed counts to: dedup_removed_by_sourceDatabase.csv')
    
    removed.head(50).to_parquet('All_data/dedup_removed_sample.parquet', index=False)
    print('Saved sample removed rows to: All_data/dedup_removed_sample.parquet')
else:
    print("\nNo cross-database duplicates found.")

Original merged dataset: 4372899 rows
After deduplication: 4065017 rows
Rows deduplicated: 307882
Cross-database duplicates found: 587819
Within-database records preserved: 2709635
Saved deduplicated dataset to: All_data/merged_all_deduplicated.parquet

Removed counts by sourceDatabase:
  sourceDatabase  removed_count
0           GBIF         286884
1           OBIS          20998
Saved removed counts to: dedup_removed_by_sourceDatabase.csv
Saved sample removed rows to: All_data/dedup_removed_sample.parquet


In [ ]:
# Compare gear and individualCount coverage between removed and kept cross-database records
comparison_cols = ['gear', 'individualCount']

if 'removed' not in globals() or 'cross_db_duplicates_deduped' not in globals():
    print('Run the cross-database deduplication cell first.')
else:
    def presence_rate(df, col):
        if col not in df.columns or len(df) == 0:
            return 0, 0.0
        present = (
            df[col]
            .astype('string')
            .str.strip()
            .replace({'': pd.NA})
            .notna()
        )
        return int(present.sum()), float(present.mean() * 100)

    kept_rows = cross_db_duplicates_deduped

    comparison_rows = []
    for col in comparison_cols:
        removed_count, removed_pct = presence_rate(removed, col)
        kept_count, kept_pct = presence_rate(kept_rows, col)
        comparison_rows.append({
            'column': col,
            'removed_present': removed_count,
            'removed_total': len(removed),
            'removed_percent': removed_pct,
            'kept_present': kept_count,
            'kept_total': len(kept_rows),
            'kept_percent': kept_pct,
            'higher_coverage': 'removed' if removed_pct > kept_pct else 'kept' if kept_pct > removed_pct else 'tie'
        })

    coverage_df = pd.DataFrame(comparison_rows)
    print(coverage_df.to_string(index=False))
    print('\nSummary:')
    for _, row in coverage_df.iterrows():
        print(f"{row['column']}: {row['higher_coverage']} records have higher coverage ({row['removed_percent']:.2f}% removed vs {row['kept_percent']:.2f}% kept)")

         column  removed_present  removed_total  removed_percent  kept_present  kept_total  kept_percent higher_coverage
           gear           164750         307882        53.510761         24530      279937      8.762686         removed
individualCount            59813         307882        19.427248        109246      279937     39.025209            kept

Summary:
gear: removed records have higher coverage (53.51% removed vs 8.76% kept)
individualCount: kept records have higher coverage (19.43% removed vs 39.03% kept)


In [27]:
# Remove within-database duplicates using aphiaID
# Prefer records that have values in individualCount, biomass, gear, and areaSampled
# when choosing the survivor for a duplicated within-database key.

# Load the deduplicated dataset
merged_deduped = pd.read_parquet('All_data/merged_all_dedup_between.parquet').copy()
if '_row_id' not in merged_deduped.columns:
    merged_deduped['_row_id'] = merged_deduped.index.astype('int64')

# Normalize sourceDatabase if not already done
if 'sourceDatabase_norm' not in merged_deduped.columns:
    merged_deduped['sourceDatabase_norm'] = (
        merged_deduped['sourceDatabase'].astype('string')
              .str.strip()
              .str.upper()
    )
    merged_deduped['sourceDatabase_norm'] = merged_deduped['sourceDatabase_norm'].replace({'': pd.NA, 'NAN': pd.NA})

# Define within-database deduplication columns
within_db_cols = ['sourceDatabase_norm', 'year', 'startDayOfYear', 'decimalLongitudeStart', 'decimalLatitudeStart', 'aphiaID']

# Only rows with all relevant columns populated are eligible for within-database dedup
required_numeric_cols = ['year', 'startDayOfYear', 'decimalLongitudeStart', 'decimalLatitudeStart']
required_text_cols = ['sourceDatabase_norm', 'aphiaID']

numeric_complete = merged_deduped[required_numeric_cols].notna().all(axis=1)
text_complete = merged_deduped[required_text_cols].apply(
    lambda s: s.astype('string').str.strip().notna() & (s.astype('string').str.strip() != '')
).all(axis=1)
within_db_mask = numeric_complete & text_complete

# Split into rows eligible for within-database dedup and rows to keep as-is
to_deduplicate_within = merged_deduped[within_db_mask].copy()
to_keep_within = merged_deduped[~within_db_mask].copy()

# Prefer records that have values in individualCount, biomass, gear, and areaSampled
# when choosing the survivor for a duplicated within-database key.
preference_cols = ['individualCount', 'biomass', 'gear', 'areaSampled']
preference_present_cols = [col for col in preference_cols if col in to_deduplicate_within.columns]
preference_scores = pd.Series(0, index=to_deduplicate_within.index, dtype='int64')
for col in preference_present_cols:
    present = (
        to_deduplicate_within[col]
        .astype('string')
        .str.strip()
        .replace({'': pd.NA})
        .notna()
        .astype('int64')
    )
    preference_scores = preference_scores.add(present, fill_value=0).astype('int64')

to_deduplicate_within['_preference_score'] = preference_scores

# Remove within-database duplicates (same sourceDatabase, place, date, and aphiaID)
# Keep the row with the most preferred observation fields; ties fall back to first occurrence after stable sorting
sort_cols = ['_preference_score', '_row_id']
sort_ascending = [True] * len(within_db_cols) + [False, True]
to_deduplicate_within = to_deduplicate_within.sort_values(
    by=within_db_cols + sort_cols,
    ascending=sort_ascending,
    kind='mergesort'
)
to_deduplicate_within_deduped = to_deduplicate_within.drop_duplicates(subset=within_db_cols, keep='first')

# Report what was removed via stable row id
kept_row_ids = set(to_deduplicate_within_deduped['_row_id'].tolist())
removed_within = to_deduplicate_within[~to_deduplicate_within['_row_id'].isin(kept_row_ids)].copy()

# Concatenate back together
to_deduplicate_within_deduped = to_deduplicate_within_deduped.drop(columns=['_preference_score'])
merged_within_deduped = pd.concat([to_deduplicate_within_deduped, to_keep_within], ignore_index=True)
if '_row_id' in merged_within_deduped.columns:
    merged_within_deduped = merged_within_deduped.drop(columns=['_row_id'])

print(f"Before within-database deduplication: {len(merged_deduped)} rows")
print(f"Eligible rows (all relevant columns present): {len(to_deduplicate_within)}")
print(f"After within-database deduplication: {len(merged_within_deduped)} rows")
print(f"Rows removed: {len(merged_deduped) - len(merged_within_deduped)}")

# Save the result
merged_within_deduped.to_parquet('All_data/merged_all_dedup_within.parquet', index=False)
print(f"Saved deduplicated dataset to: All_data/merged_all_dedup_within.parquet")

if len(removed_within) > 0:
    print(f"\nWithin-database duplicates removed by sourceDatabase:")
    print(removed_within['sourceDatabase_norm'].value_counts(dropna=False))
    print(f"\nExample removed rows:")
    print(removed_within.head(10)[['sourceDatabase_norm', 'year', 'startDayOfYear', 'decimalLongitudeStart', 'decimalLatitudeStart', 'aphiaID']])
else:
    print("\nNo within-database duplicates found.")

Before within-database deduplication: 2945953 rows
Eligible rows (all relevant columns present): 1978528
After within-database deduplication: 2945953 rows
Rows removed: 0
Saved deduplicated dataset to: All_data/merged_all_deduplicated.parquet

No within-database duplicates found.
